# Gemma Vision 2.0: On-Device Multimodal Accessibility for Blind & Low-Vision Users

**Gemma 4 Good Hackathon Submission** | Track: Digital Equity & Inclusivity | Prize Targets: Main + Digital Equity + LiteRT

---

## Overview

Gemma Vision 2.0 is a fully on-device, multimodal accessibility companion for blind and low-vision (BLV) users. It runs **Gemma 4 E2B** directly on Android via Google's **LiteRT-LM** runtime — no cloud, no internet, no data leaving the device.

The app captures a photo, sends it through the Gemma 4 vision encoder, and streams a spoken description to the user via Text-to-Speech. Seven specialized tool-calling functions handle structured tasks like currency identification, barcode scanning, document reading, and color identification.

### Why This Matters

- **285 million people** worldwide have visual impairments (WHO 2024)
- **90% live in low-income regions** where cloud-dependent solutions fail
- Existing solutions (Be My Eyes, Seeing AI) require internet connectivity
- On-device inference preserves privacy — no images leave the phone

### Technical Stack

| Component | Technology |
|-----------|------------|
| Model | Gemma 4 E2B (2.4 GB, INT4 quantized) |
| Runtime | LiteRT-LM 0.11.0 (`.litertlm` format) |
| Backend | CPU via XNNPACK (GPU/NPU on real devices) |
| Camera | CameraX 1.4 with ImageProxy.toBitmap() |
| TTS | Android TextToSpeech with streaming chunking |
| Wake Word | Picovoice Porcupine |
| Tools | ML Kit (OCR, Translation, Barcode), OpenFoodFacts API |
| Language | Kotlin 2.2, Compose, minSdk 31 |

## Architecture

```
                    ┌───────────────────────────────────────┐
                    │         INPUT LAYER                │
                    │  Wake Word  |  Gamepad  |  ADB      │
                    └─────────────────┬─────────────────────┘
                                  │ VisionIntent
                                  ▼
                    ┌───────────────────────────────────────┐
                    │       ORCHESTRATION LAYER           │
                    │  MainActivity.runOneTurn()           │
                    │  300ms debounce • MAX_HOPS=4         │
                    └─────────┬───────────────────┬─────────┘
                              │                   │
                   ┌─────────▼───────┐  ┌───────▼────────┐
                   │  Camera Capture  │  │ System Prompt  │
                   │  CameraX 1.4     │  │ + Tool Schemas │
                   │  → PNG bytes     │  │ + User Intent  │
                   └────────┬────────┘  └───────┬────────┘
                           │                  │
                           ▼                  ▼
                    ┌───────────────────────────────────────┐
                    │        INFERENCE ENGINE              │
                    │  Gemma 4 E2B via LiteRT-LM           │
                    │  Content.ImageBytes + Content.Text    │
                    │  → Streaming token output            │
                    └─────────┬───────────────────┬─────────┘
                              │                   │
              ┌─────────────▼────┐    ┌──────▼────────────┐
              │  Tool Call Router │    │  Streaming TTS    │
              │  <gv:tool_call>   │    │  Sentence-boundary│
              │  Parse → Dispatch │    │  chunking + speak │
              └──────┬────────────┘    └───────────────────┘
                     │
    ┌─────────────────────────────────────────────────┐
    │              7 TOOL IMPLEMENTATIONS                   │
    │  identify_currency  • read_document  • scan_barcode    │
    │  describe_scene     • identify_color • translate_text  │
    │  system_action (dial, text, alarm, email)              │
    └─────────────────────────────────────────────────┘
```

## Research Foundations

Our design decisions are grounded in peer-reviewed BLV research:

| Paper | Venue | Key Insight | How We Use It |
|-------|-------|-------------|---------------|
| VizWiz Grand Challenge (Gurari et al.) | CVPR 2018 | 24% of BLV photos are unanswerable; models must learn to abstain | System prompt requires explicit abstention on blurry/occluded images |
| Context-Aware Image Descriptions (Stangl & Morris) | ASSETS 2021 | BLV users want descriptions tailored to their current task | We inject foreground-app context, locale, and intent into prompts |
| SPHERE: Spatial Reasoning Benchmark | ACL 2025 | VLMs systematically fail left/right spatial reasoning | System prompt forbids confident spatial claims unless visually obvious |
| Misfitting With AI (Akter et al.) | ASSETS 2024 | BLV users distrust AI that sounds confident but is wrong | We never adjudicate safety; we describe, user decides |
| OCRBench v2 | NeurIPS 2025 | Multi-page OCR remains challenging for all VLMs | We use ML Kit TextRecognition as a reconciliation pipeline |

## Key Innovation: Multi-Hop Tool Re-Injection

Gemma Vision 2.0 implements a novel **multi-hop tool-calling loop** that lets the model chain multiple tools in a single user request:

1. User presses a button or speaks a wake word
2. Camera captures a frame, preprocessed per intent (contrast boost for OCR, etc.)
3. System prompt + tool schemas + image sent to Gemma 4 via `sendMessageAsync`
4. Model streams tokens. If it emits `<gv:tool_call>{...}</gv:tool_call>`, the router:
   - Parses the JSON tool call
   - Dispatches to the appropriate handler (ML Kit, network API, etc.)
   - Appends the call + response to the transcript
   - Re-invokes `generateStream` for the model to continue
5. Repeat up to 4 hops. Text before tool calls is spoken immediately via TTS.

This protocol uses custom XML tags (`<gv:tool_call>`) rather than Gemma 4's native asymmetric tokens, because the custom format is more reliable under streaming conditions and easier to parse incrementally.

## Inference Proof: Gemma 4 E2B Running On-Device

We verified end-to-end inference on an Android emulator (Pixel 9 Pro XL, API 35). The model generates coherent text via LiteRT-LM's CPU backend with XNNPACK delegates.

### Logcat Evidence

```
I MediaPipeProvider: Using model: gemma-4-E2B-it.litertlm (external, 2588147712 bytes)
I MediaPipeProvider: LiteRT-LM Engine ready: text=CPU(numOfThreads=null) visionGpu=false
I MediaPipeProvider: LiteRT-LM cold load: 3453ms
I GemmaVisionApp: Inference ready: MediaPipeProvider

I MainActivity: Debug intent: DebugTextOnly
I MainActivity: runOneTurn: intent=DebugTextOnly requiresImage=false
I MainActivity: Prompt length: 55 chars
I MediaPipeProvider: sendMessageAsync: prompt.len=55 hasImage=false

D MediaPipeProvider: onMessage: class=Message len=5 chunk=Hello
D MediaPipeProvider: onMessage: class=Message len=1 chunk=,
D MediaPipeProvider: onMessage: class=Message len=2 chunk= I
D MediaPipeProvider: onMessage: class=Message len=3 chunk= am
D MediaPipeProvider: onMessage: class=Message len=6 chunk= happy
D MediaPipeProvider: onMessage: class=Message len=3 chunk= to
D MediaPipeProvider: onMessage: class=Message len=7 chunk= assist
D MediaPipeProvider: onMessage: class=Message len=4 chunk= you
D MediaPipeProvider: onMessage: class=Message len=1 chunk=!
I MediaPipeProvider: onDone — generation finished
```

**Output: "Hello, I am happy to assist you!"**

> Note: On emulator CPU, inference is ~2.7 min/token. On a real Pixel 9 Pro with GPU/NPU, expected performance is 15-20 tok/sec (sub-second response time).

## Model Loading Performance

| Scenario | Load Time | Notes |
|----------|-----------|-------|
| First cold load (no cache) | ~15 min | XNNPACK weight cache built from scratch (788 MB) |
| Warm load (XNNPACK cached) | **3.4 sec** | Cache persisted across restarts |
| GPU attempted, fallback to CPU | +2 min | GPU (OpenGL) fails on emulator, succeeds on real devices |
| CPU-first (our optimization) | **3.4 sec** | Skip GPU probe entirely on emulator |

The XNNPACK weight cache stores pre-compiled delegate kernels for all 976 subgraphs across 5 model sections:
- Main decoder: 788 MB
- Vision encoder: 217 MB  
- Vision adapter: 4.7 MB
- Audio encoder: 91 MB
- Audio adapter: 9.4 MB

## Code Walkthrough

### 1. LiteRT-LM Engine Initialization

```kotlin
// MediaPipeProvider.kt - CPU-first backend selection
engine = tryEngine(modelFile, Backend.CPU(), visionGpu = false)
    ?: tryEngine(modelFile, Backend.GPU(), visionGpu = true)
    ?: throw IllegalStateException("Engine failed on both backends")

conversation = engine!!.createConversation(
    ConversationConfig(
        samplerConfig = SamplerConfig(topK = 40, topP = 0.95, temperature = 0.7),
        systemInstruction = null,  // rendered per-turn for context
        tools = emptyList(),       // custom tool protocol via system prompt
    )
)
```

### 2. Streaming Inference with Vision

```kotlin
// Multimodal content: image bytes + text prompt
val contents = mutableListOf<Content>()
image?.let { contents.add(Content.ImageBytes(it.toPngByteArray())) }
contents.add(Content.Text(prompt))

conv.sendMessageAsync(Contents.of(contents), callback, emptyMap())
```

### 3. Multi-Hop Tool Loop

```kotlin
// MainActivity.runOneTurn() - stream tokens, detect tool calls, re-inject
while (hops < MAX_HOPS) {
    provider.generateStream(transcript, image).collect { chunk ->
        buffer.append(chunk)
        val safe = router.safePrefixToSpeak(buffer.toString())
        if (safe.isNotEmpty()) tts.feed(safe)  // speak immediately
        
        val call = router.tryParse(buffer.toString())
        if (call != null) pendingCall = call
    }
    if (pendingCall == null) { tts.flush(); return }  // plain answer
    
    // Tool dispatch + re-prompt
    val response = router.dispatch(call, image)
    transcript += router.encodeCall(call) + router.encodeResponse(response)
    hops++
}
```

### 4. Streaming TTS with Sentence Boundaries

```kotlin
// StreamingTts.kt - flush at sentence boundaries for natural speech
fun feed(text: String) {
    synchronized(buffer) {
        buffer.append(text)
        while (true) {
            val boundary = findBoundary(buffer) ?: break
            val sentence = buffer.substring(0, boundary + 1).trim()
            if (sentence.isNotEmpty()) speak(sentence)
            buffer.delete(0, boundary + 1)
        }
    }
}
```

## Tool Implementations

| Tool | Purpose | Backend | Key Design Choice |
|------|---------|---------|-------------------|
| `identify_currency` | Identify currency denomination from photo | Gemma 4 re-prompt | Specialized prompt focuses model on denomination + side |
| `read_document` | OCR text from documents, signs, menus | ML Kit TextRecognition | Reading-order reconstruction from bounding boxes |
| `scan_barcode_then_lookup` | Scan barcode → lookup product name | ML Kit Barcode + OpenFoodFacts | Falls back to OCR if no barcode found |
| `describe_scene` | Rich scene description with context | Gemma 4 (context-aware) | Injects foreground-app context per Stangl et al. |
| `identify_color` | Dominant color identification | Histogram bucket analysis | Fast on-device; no model call needed |
| `translate_text` | Translate visible text to user's language | ML Kit Translation | On-device translation models (~30 MB each) |
| `system_action` | Dial phone, send SMS, set alarm, email | Android Intent dispatch | ACTION_DIAL, ACTION_SENDTO, AlarmClock |

## Input Modalities

Gemma Vision 2.0 supports three input methods, designed for different BLV contexts:

1. **Wake Word** ("Hey Gemma") — Picovoice Porcupine, always-on foreground service
2. **8BitDo Micro Gamepad** — 6 buttons mapped to 6 intent categories via Bluetooth HID
3. **ADB Debug Intent** — `adb shell am broadcast` for development/testing

The gamepad approach is novel: the 8BitDo Micro ($20, 45×25mm) fits in a pocket and provides tactile, eyes-free access to all functions. Each button triggers a different `VisionIntent` with tuned image resolution and prompt scaffolding.

## Privacy & Offline-First Design

- **100% on-device inference** — Gemma 4 E2B runs locally via LiteRT-LM
- **No cloud dependency** — works in airplane mode, rural areas, disaster zones
- **No image upload** — photos never leave the device
- **ML Kit runs locally** — OCR, translation, barcode scanning all on-device
- **Only exception**: OpenFoodFacts barcode lookup (optional, degrades gracefully)

This design is critical for the 90% of visually impaired people in low-income countries (WHO) where internet access is unreliable and data privacy concerns are heightened.

## Accessibility Service Integration

The optional `GemmaVisionA11yService` reads the foreground app's package name to provide context-aware descriptions:

- If the user is in a **messaging app**, descriptions prioritize reading text
- If in a **shopping app**, descriptions focus on product details
- If in a **navigation app**, spatial descriptions are more cautious

This follows Stangl et al.'s (ASSETS 2021) finding that BLV users strongly prefer context-dependent image descriptions over generic ones.

## Comparison with Gemma Vision 1.0

| Feature | v1.0 (Gemma 3n) | v2.0 (Gemma 4 E2B) |
|---------|-----------------|---------------------|
| Model | Gemma 3n (nano) | **Gemma 4 E2B** (2.4B params) |
| Modalities | Vision + Text | **Vision + Audio + Text** (trimodal) |
| Tool Calling | None | **7 tools with multi-hop re-injection** |
| Runtime | MediaPipe Tasks GenAI | **LiteRT-LM** (.litertlm format) |
| Streaming | Basic | **Sentence-boundary TTS streaming** |
| Wake Word | None | **Picovoice Porcupine** |
| Input | Touch only | **Gamepad + Wake Word + Touch** |
| Context | None | **A11y service + locale + app context** |
| Research Grounding | Minimal | **5 peer-reviewed BLV papers** |
| Safety | Generic | **Research-backed abstention + no safety adjudication** |

## Build & Run

### Prerequisites
- Android Studio with JDK 21 (bundled JBR)
- Android SDK 35, minSdk 31
- Gemma 4 E2B model file (~2.4 GB)

### Steps

```bash
# 1. Clone the repository
git clone https://github.com/<repo>/gemma-vision-2.git
cd gemma-vision-2

# 2. Download the model (requires Kaggle/HF auth)
# Place gemma-4-E2B-it.litertlm in models/

# 3. Build the APK
export JAVA_HOME="/path/to/jdk21"
./gradlew :app:installDebug

# 4. Push the model to device/emulator
adb push models/gemma-4-E2B-it.litertlm \
    /sdcard/Android/data/org.cmu.gemmavision2/files/

# 5. Launch and wait for model load
adb shell am start -n org.cmu.gemmavision2/.MainActivity

# 6. Test inference
adb shell am broadcast \
    -a org.cmu.gemmavision2.TEST_INTENT \
    --es intent DescribeScene \
    -p org.cmu.gemmavision2
```

## Impact Statement

Gemma Vision 2.0 demonstrates that **state-of-the-art multimodal AI can run entirely on a phone** — no cloud, no internet, no data leaving the device. For the 285 million visually impaired people worldwide, this means:

- **Access anywhere**: Works in rural areas, disaster zones, and developing regions without internet
- **Privacy preserved**: Medical labels, financial documents, and personal photos never leave the device
- **Cost**: Free after device purchase — no subscription, no API fees
- **Dignity**: Users don't need to share their visual world with a cloud service or stranger

The combination of Gemma 4's multimodal capabilities with LiteRT-LM's efficient on-device runtime makes this the first truly private, truly offline, truly capable visual assistant for BLV users.

---

*Built at Carnegie Mellon University | Gemma 4 Good Hackathon 2026*

*Contact: bnallamo@andrew.cmu.edu*